# FC Tokyo Feature Engineering and OLS

FC東京ホームゲームの特徴量を作成し、線形回帰モデルを比較する。


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "requirements.txt").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import re


In [ ]:
df = pd.read_csv(RAW_DATA_DIR / "J1_tokyo_home_2015-2024.csv")


In [ ]:
df["コロナ禍ダミー"] = df["シーズン"].apply(lambda x: 1 if x in [2020, 2021] else 0)


In [ ]:
df["国立フラグ"] = df["スタジアム"].str.contains("国立").astype(int)


In [ ]:
# 試合日カラムをstr型に変換し､曜日を抽出
df["試合日"] = df["試合日"].astype(str)

df["曜日"] = df["試合日"].str.extract(r"[（(]([^)）]+)[)）]")
# 土日祝に該当するかどうかを判定
df["休日フラグ"] = df["曜日"].str.contains("土|日|祝").astype(int)
df


### アウェイのみを説明変数としたとき


In [ ]:
X = pd.get_dummies(df["アウェイ"], drop_first=True).astype(float)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


### アウェイとコロナ禍を説明変数としたとき


In [ ]:
# コロナ禍ダミー(2020､2021=1)を追加し､重回帰分析
df["コロナ禍ダミー"] = df["シーズン"].apply(lambda x: 1 if x in [2020, 2021] else 0)
X = pd.get_dummies(df[["アウェイ", "コロナ禍ダミー"]], drop_first=True).astype(float)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)
model = sm.OLS(y, X).fit()
print(model.summary())


### アウェイ､曜日､コロナ禍を説明変数としたとき


In [ ]:
# アウェイ､曜日､コロナ禍ダミーを説明変数とした重回帰分析
X = pd.concat(
    [
        pd.get_dummies(df["アウェイ"], drop_first=True).astype(float),
        df[["休日フラグ", "コロナ禍ダミー"]].astype(float),
    ],
    axis=1,
)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


### アウェイ､曜日､コロナ禍､国立を説明変数としたとき


In [ ]:
# アウェイ､曜日､コロナ禍ダミー､国立フラグを説明変数とした重回帰分析
X = pd.concat(
    [
        pd.get_dummies(df["アウェイ"], drop_first=True).astype(float),
        df[["休日フラグ", "コロナ禍ダミー", "国立フラグ"]].astype(float),
    ],
    axis=1,
)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


### コロナ禍ダミーのみを説明変数としたとき


In [ ]:
# コロナ禍ダミーのみを説明変数とした重回帰分析
X = df[["コロナ禍ダミー"]].astype(float)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
df["試合日"].unique()[:10]


In [ ]:
# 試合日から日付部分を抽出
df["date_str"] = df["試合日"].str.extract(r"(\d{2}/\d{2}/\d{2})")[0]

# K/O時刻を文字列に変換（null値対策）
df["K/O時刻"] = df["K/O時刻"].astype(str)

# 日付とK/O時刻を組み合わせてdatetime型に変換
df["datetime"] = pd.to_datetime(
    df["date_str"] + " " + df["K/O時刻"], format="%y/%m/%d %H:%M", errors="coerce"
)

# 日付のみのカラムも保持
df["date"] = pd.to_datetime(df["date_str"], format="%y/%m/%d", errors="coerce")

# datetimeから年を抽出
df["year"] = df["datetime"].dt.year

# datetimeから時間を抽出（14:00なら14）
df["hour"] = df["datetime"].dt.hour

# 一時カラムを削除
df.drop(columns=["date_str"], inplace=True)


In [ ]:
df_weather = pd.read_csv(RAW_DATA_DIR / "weather_tokyo_2015-2024.csv", header=4)
df_weather


In [ ]:
# df_weatherの0列目､1列目､4列目を抽出し､列目をdate, temperature,rainに変更
df_weather = df_weather.iloc[:, [0, 1, 4]]
df_weather.columns = ["date", "temperature", "rain"]
df_weather


In [ ]:
# df_weatherのdateをdatetime型に変換
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y/%m/%d")
df_weather


In [ ]:
df


In [ ]:
# 読み込み
df = pd.read_csv(RAW_DATA_DIR / "J1_tokyo_home_2015-2024.csv")
# 各変数
df["コロナ禍ダミー"] = df["シーズン"].apply(lambda x: 1 if x in [2020, 2021] else 0)
df["国立フラグ"] = df["スタジアム"].str.contains("国立").astype(int)
df["試合日"] = df["試合日"].astype(str)
df["曜日"] = df["試合日"].str.extract(r"[（(]([^)）]+)[)）]")
df["休日フラグ"] = df["曜日"].str.contains("土|日|祝").astype(int)
df["rolling_mean_3"] = df["入場者数"].rolling(window=3).mean()

# 試合日から日付部分を抽出し、K/O時刻と組み合わせてdatetime型に変換
df["date_str"] = df["試合日"].str.extract(r"(\d{2}/\d{2}/\d{2})")[0]
df["K/O時刻"] = df["K/O時刻"].astype(str)
df["datetime"] = pd.to_datetime(
    df["date_str"] + " " + df["K/O時刻"], format="%y/%m/%d %H:%M", errors="coerce"
)
df["date"] = pd.to_datetime(df["date_str"], format="%y/%m/%d", errors="coerce")
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["hour"] = df["datetime"].dt.hour
df.drop(columns=["date_str"], inplace=True)


# dateをキーにしてdfのマージ
df_merged = pd.merge(df, df_weather, on="date", how="left")
df_merged.head()


### 気温プラス雨が降ったか否か


In [ ]:
df_merged["temp_zone"] = pd.cut(
    df_merged["temperature"],
    bins=[0, 10, 15, 20, 25, 30, 35, 40],
    labels=["寒い", "やや寒い", "快適", "快適2", "暑い", "猛暑", "酷暑"],
)

df_merged["rain_flag"] = df_merged["rain"].apply(lambda x: 1 if x > 0 else 0)


In [ ]:
# アウェイ､曜日､コロナ禍ダミー､気温､雨
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        pd.get_dummies(df_merged["temp_zone"], drop_first=True).astype(float),
        df_merged[["休日フラグ", "コロナ禍ダミー", "国立フラグ", "rain_flag"]].astype(
            float
        ),
    ],
    axis=1,
)
X = sm.add_constant(X)
y = df["入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
df_merged["lag1"] = df_merged["入場者数"].shift(1)
df_merged["lag2"] = df_merged["入場者数"].shift(2)
df_merged["rolling_mean_2"] = df_merged["入場者数"].rolling(window=2).mean()
df_merged["rolling_mean_5"] = df_merged["入場者数"].rolling(window=5).mean()
df_merged["rolling_mean_3"] = df_merged["入場者数"].rolling(window=3).mean()
df_merged["rolling_mean_7"] = df_merged["入場者数"].rolling(window=7).mean()


In [ ]:
df_merged.head()


### 前試合に勝ったか否か


In [ ]:
df_merged


In [ ]:
df_merged[["home_score", "away_score"]] = (
    df_merged["スコア"].str.split("-", expand=True).astype(int)
)


In [ ]:
#  勝った場合が1､負けと引き分けが0
df_merged["result_numeric"] = np.select(
    [
        df_merged["home_score"] > df_merged["away_score"],
        df_merged["home_score"] == df_merged["away_score"],
        df_merged["home_score"] < df_merged["away_score"],
    ],
    [1, 0, 0],
    default=0,
)


In [ ]:
# 前の試合､勝ったか否かのフラグ
df_merged["lag1_result_numeric"] = df_merged["result_numeric"].shift(1)
df_merged.head()


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            ["コロナ禍ダミー", "国立フラグ", "休日フラグ", "lag1_result_numeric"]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df_merged.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


### rolling_mean, lag系


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            ["コロナ禍ダミー", "国立フラグ", "休日フラグ", "lag1", "rain_flag"]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df_merged.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            [
                "コロナ禍ダミー",
                "国立フラグ",
                "休日フラグ",
                "rolling_mean_2",
                "rain_flag",
            ]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df_merged.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            [
                "コロナ禍ダミー",
                "国立フラグ",
                "休日フラグ",
                "rolling_mean_3",
                "rain_flag",
            ]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df_merged.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# アウェイチームの係数のp-valueの図示
# モデルのp-valueを取得
plt.rcParams["font.family"] = "Hiragino Sans"

pvalues = model.pvalues

# アウェイチーム（チーム名）のみを抽出（const、コロナ禍ダミー等を除外）
exclude_vars = [
    "const",
    "コロナ禍ダミー",
    "国立フラグ",
    "休日フラグ",
    "rolling_mean_3",
    "rolling_mean_5",
    "rain_flag",
    "rolling_mean_2",
]
team_pvalues = pvalues.drop(
    labels=[v for v in exclude_vars if v in pvalues.index], errors="ignore"
)

# p-valueでソート（小さい順）
team_pvalues_sorted = team_pvalues.sort_values()

# 可視化
fig, ax = plt.subplots(figsize=(10, 8))


# カラーマップ：p < 0.05 は緑、0.05 ≤ p < 0.10 は黄色、それ以外は赤
def get_color(p):
    if p < 0.05:
        return "green"
    else:
        return "salmon"


colors = [get_color(p) for p in team_pvalues_sorted.values]

# 横棒グラフで表示
bars = ax.barh(
    range(len(team_pvalues_sorted)), team_pvalues_sorted.values, color=colors
)

# y軸のラベルをチーム名に
ax.set_yticks(range(len(team_pvalues_sorted)))
ax.set_yticklabels(team_pvalues_sorted.index)

# p = 0.05の基準線
ax.axvline(x=0.05, color="blue", linestyle="--", linewidth=2, label=" p = 0.05 ")

# 凡例用のダミー（色の説明）
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="green", label="有意 ( p < 0.05)"),
    Patch(facecolor="salmon", label="有意でない ( p => 0.05)"),
    plt.Line2D([0], [0], color="blue", linestyle="--", linewidth=2, label=" p = 0.05 "),
]

# ラベルとタイトル
ax.set_xlabel("p-value", fontsize=12)
ax.set_ylabel("アウェイチーム", fontsize=12)
ax.set_title("アウェイチーム係数のp-value（入場者数への影響）", fontsize=14)
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

# グリッド追加
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# 統計的に有意なチーム（p < 0.05）を表示
print("\n【有意】p < 0.05 のアウェイチーム:")
significant_teams = team_pvalues[team_pvalues < 0.05].sort_values()
for team, p in significant_teams.items():
    coef = model.params[team]
    print(f"  {team}: p-value = {p:.4f}, 係数 = {coef:+.1f}")


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            [
                "コロナ禍ダミー",
                "国立フラグ",
                "休日フラグ",
                "rolling_mean_5",
                "rain_flag",
            ]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df_merged.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


### 説明変数全部込み


In [ ]:
X = pd.concat(
    [
        pd.get_dummies(df_merged["アウェイ"], drop_first=True).astype(float),
        df_merged[
            [
                "コロナ禍ダミー",
                "国立フラグ",
                "休日フラグ",
                "rolling_mean_3",
                "rain_flag",
            ]
        ].astype(float),
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)
y = df.loc[X.index, "入場者数"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
df_merged.to_csv(PROCESSED_DATA_DIR / "to_ML.csv", index=False)


### 日付系(year, month, hour)


In [ ]:
# 説明変数
X = pd.concat(
    [
        pd.get_dummies(df["アウェイ"], drop_first=True).astype(float),
        df[
            [
                "コロナ禍ダミー",
                "国立フラグ",
                "休日フラグ",
                "rolling_mean_3",
                "year",
                "month",
                "hour",
            ]
        ],
    ],
    axis=1,
).dropna()
X = sm.add_constant(X)

# 目的変数
y = df.loc[X.index, "入場者数"].astype(float)


# モデル
model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
df_merged
